In [1]:
# Single-run Mind2Web + SoM inference from public Google Drive input folder
import json
import re
import sys
import os
import shutil
import importlib.util
import subprocess
import time
from pathlib import Path

# -----------------------------
# 0) Optional dependency install
# -----------------------------
AUTO_INSTALL_DEPS = True
required_modules = {
    "numpy": "numpy",
    "easyocr": "easyocr",
    "gdown": "gdown",
    "PIL": "pillow",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "open_clip": "open_clip_torch",
    "ultralytics": "ultralytics",
    "cv2": "opencv-python-headless",
}
missing_packages = [pkg for mod, pkg in required_modules.items() if importlib.util.find_spec(mod) is None]

if AUTO_INSTALL_DEPS and missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

if AUTO_INSTALL_DEPS:
    print("Installing Magma-compatible transformers fork...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/jwyang/transformers.git@dev/jwyang-v4.48.2",
    ])

import torch
import torch.nn as nn
import numpy as np
import easyocr
import gdown
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from peft import PeftModel
from transformers import AutoConfig, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

# -----------------------------
# 1) Runtime / compatibility patches
# -----------------------------
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

if not hasattr(torch, "_original_sum_backup"):
    torch._original_sum_backup = torch.sum


def _patched_sum(input, *args, **kwargs):
    if isinstance(input, bool):
        input = torch.tensor(input, dtype=torch.long)
    elif isinstance(input, torch.Tensor) and input.dtype == torch.bool and (len(args) > 0 or "dim" in kwargs):
        input = input.long()
    return torch._original_sum_backup(input, *args, **kwargs)


torch.sum = _patched_sum
print("Applied torch.sum bool-tensor compatibility patch")

# -----------------------------
# 2) Config
# -----------------------------
BASE_MODEL = "microsoft/Magma-8B"
CHECKPOINT_GDRIVE_URL = "https://drive.google.com/drive/folders/1_Wp2wBEH1-aUt9QzMmcDVCIIbLw11fR_?usp=drive_link"
CHECKPOINT_LOCAL_DIR = Path("./checkpoints/mind2web_adapter")

# Public input folder containing input.png and prompt.txt
INPUT_FOLDER_URL = "https://drive.google.com/drive/folders/1zBAf-dJRi38sTrLBxV-RrXaY-MLezESm"
MODEL_IMAGE_SIZE = 640
FORCE_REDOWNLOAD_CHECKPOINT = False
WORK_DIR = Path.cwd().resolve() / "full"
LOCAL_INPUT_DIR = WORK_DIR / "drive_input"
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR = WORK_DIR / "output"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXECUTOR_OUTPUT_DIR = WORK_DIR / "drive_sync" / "output"
EXECUTOR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INSTRUCTION_TEMPLATE = (
    "Imagine that you are imitating humans doing web navigation for a task step by step. "
    "At each stage, you can see the webpage like humans by a screenshot and know the previous "
    "actions before the current step decided by yourself through recorded history. You need to "
    "decide on the following action to take. You can click an element with the mouse, select an "
    "option, or type text with the keyboard. The output format should be a dictionary like: \n"
    "{\"ACTION\": \"CLICK\" or \"TYPE\" or \"SELECT\", \"MARK\": a numeric id, e.g., 5, "
    "\"VALUE\": a string value for the action if applicable, otherwise None}.\n"
    "You are asked to complete the following task: {task_prompt}\n"
    "For your convenience, I have labeled the candidates with numeric marks and bounding boxes "
    "on the screenshot. What is the next action you would take?"
)

# -----------------------------
# 3) Helpers
# -----------------------------
def parse_action(text: str):
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
            return parsed[0]
    except json.JSONDecodeError:
        pass

    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except Exception:
            pass

    return {"raw_response": text, "parse_error": True}


def _pick_latest_file(root: Path, name: str) -> Path:
    candidates = [p for p in root.rglob(name) if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"{name} not found after downloading folder")
    return max(candidates, key=lambda p: p.stat().st_mtime_ns)


def wait_for_inputs_from_public_drive(folder_url: str, local_root: Path, poll_seconds: int = 5):
    local_root.mkdir(parents=True, exist_ok=True)
    while True:
        try:
            print("Downloading input folder snapshot from public Drive...")
            gdown.download_folder(
                url=folder_url,
                output=str(local_root),
                quiet=False,
                use_cookies=False,
                remaining_ok=True,
            )
            image_path = _pick_latest_file(local_root, "input.png")
            prompt_path = _pick_latest_file(local_root, "prompt.txt")
            return image_path, prompt_path
        except FileNotFoundError:
            print(f"input.png/prompt.txt not found yet. Retrying in {poll_seconds}s...")
            time.sleep(poll_seconds)
        except Exception as e:
            print(f"Download error: {e}. Retrying in {poll_seconds}s...")
            time.sleep(poll_seconds)


# -----------------------------
# 4) SoM OCR logic
# -----------------------------
# Define project_root if not already defined
if 'project_root' not in dir():
    # Try to find project root
    cwd = os.getcwd()
    if cwd == '/content':
        project_root = '/content/Magma'
        if not os.path.exists(project_root):
            os.system('git clone https://github.com/microsoft/Magma.git /content/Magma')
    else:
        # Search upward for Magma project
        project_root = cwd
        for _ in range(5):
            if os.path.exists(os.path.join(project_root, 'magma', 'modeling_magma.py')):
                break
            project_root = os.path.dirname(project_root)
    print(f"Project root: {project_root}")

# Add project root to path for SOM utilities
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Optional: pull a PR branch in Colab before importing SOM utilities.
# Example: PR_NUMBER = 123
PR_NUMBER = None

if os.getcwd() == '/content':
    if PR_NUMBER is not None:
        print(f"Fetching PR #{PR_NUMBER} into {project_root}...")
        subprocess.run(["git", "-C", project_root, "fetch", "origin", f"pull/{PR_NUMBER}/head:pr-{PR_NUMBER}"], check=True)
        subprocess.run(["git", "-C", project_root, "checkout", f"pr-{PR_NUMBER}"], check=True)
    else:
        subprocess.run(["git", "-C", project_root, "pull", "--ff-only"], check=False)

import importlib
import agents.ui_agent.util.som as som_module
som_module = importlib.reload(som_module)
MarkHelper = som_module.MarkHelper
plot_boxes_with_marks = som_module.plot_boxes_with_marks
print("SOM utilities loaded from project")

def _setup_new_font_colab(self, image_height, image_width):
    key = f"{image_height}_{image_width}"
    fontsize = self.min_font_size

    font_file = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    if not font_file.exists():
        print("Installing system font package: fonts-dejavu-core ...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"], check=True)

    if not font_file.exists():
        raise FileNotFoundError(f"Required system font not found: {font_file}")

    font = ImageFont.truetype(str(font_file), fontsize)
    while min(self._MarkHelper__get_markSize('555', image_height, image_width, font)) < min(
        self.max_font_size, self.max_font_proportion * min(image_height, image_width)
    ):
        fontsize += 1
        font = ImageFont.truetype(str(font_file), fontsize)

    self.font_dict[key] = font
    self.markSize_dict[key] = {
        1: self._MarkHelper__get_markSize('5', image_height, image_width, font),
        2: self._MarkHelper__get_markSize('55', image_height, image_width, font),
        3: self._MarkHelper__get_markSize('555', image_height, image_width, font),
    }

MarkHelper._setup_new_font = _setup_new_font_colab
print("Applied in-cell MarkHelper font fix (system DejaVuSans)")


def build_som_candidates(image: Image.Image):
    width, height = image.size
    image_np = np.array(image)

    print("Running EasyOCR for SoM region proposals...")
    ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)
    ocr_results = ocr_reader.readtext(image_np, text_threshold=0.5)
    print(f"Detected {len(ocr_results)} OCR regions")

    all_bboxes_normalized = []
    for idx, (coords, text, confidence) in enumerate(ocr_results):
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        x1, x2 = min(xs), max(xs)
        y1, y2 = min(ys), max(ys)

        y_norm = max(0.0, min(1.0, y1 / height))
        x_norm = max(0.0, min(1.0, x1 / width))
        h_norm = max(0.0, min(1.0 - y_norm, (y2 - y1) / height))
        w_norm = max(0.0, min(1.0 - x_norm, (x2 - x1) / width))

        all_bboxes_normalized.append((y_norm, x_norm, h_norm, w_norm))
        if idx < 10:
            print(f"[{idx}] '{text}' (conf={confidence:.2f})")

    som_annotated_image = image.copy()
    if all_bboxes_normalized:
        mark_helper = MarkHelper()
        som_annotated_image = plot_boxes_with_marks(
            som_annotated_image,
            all_bboxes_normalized,
            mark_helper,
            edgecolor=(255, 0, 0),
            linewidth=2,
            normalized_to_pixel=True,
            add_mark=True,
        )

    candidate_bboxes = {
        idx: (
            int(x_norm * width),
            int(y_norm * height),
            int((x_norm + w_norm) * width),
            int((y_norm + h_norm) * height),
        )
        for idx, (y_norm, x_norm, h_norm, w_norm) in enumerate(all_bboxes_normalized)
    }

    return som_annotated_image, candidate_bboxes


# -----------------------------
# 5) Model + adapter load
# -----------------------------
def _has_valid_checkpoint(path: Path) -> bool:
    has_config = (path / "adapter_config.json").exists()
    has_weights = (path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists()
    return has_config and has_weights


def _download_checkpoint_folder(url: str, output_dir: Path, retries: int = 3):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            print(f"Checkpoint download attempt {attempt}/{retries} (use_cookies=False)")
            gdown.download_folder(
                url=url,
                output=str(output_dir),
                quiet=False,
                use_cookies=False,
                remaining_ok=False,
            )
            return
        except Exception as exc:
            last_exc = exc
            print(f"Checkpoint download failed on attempt {attempt}: {exc}")
            if attempt < retries:
                time.sleep(5 * attempt)

    print("Retrying checkpoint download with use_cookies=True ...")
    try:
        gdown.download_folder(
            url=url,
            output=str(output_dir),
            quiet=False,
            use_cookies=True,
            remaining_ok=False,
        )
    except Exception as exc:
        raise RuntimeError(
            "Google Drive blocked automated folder download (rate-limit or anti-bot challenge). "
            "Wait and retry, or mirror the checkpoint outside Drive and update CHECKPOINT_LOCAL_DIR."
        ) from (last_exc or exc)


if FORCE_REDOWNLOAD_CHECKPOINT and CHECKPOINT_LOCAL_DIR.exists():
    print(f"Removing existing checkpoint dir: {CHECKPOINT_LOCAL_DIR}")
    shutil.rmtree(CHECKPOINT_LOCAL_DIR)
CHECKPOINT_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

if _has_valid_checkpoint(CHECKPOINT_LOCAL_DIR):
    print(f"Using cached checkpoint in {CHECKPOINT_LOCAL_DIR}")
else:
    print(f"Downloading checkpoint folder from Google Drive...\n{CHECKPOINT_GDRIVE_URL}")
    _download_checkpoint_folder(CHECKPOINT_GDRIVE_URL, CHECKPOINT_LOCAL_DIR)
    if not _has_valid_checkpoint(CHECKPOINT_LOCAL_DIR):
        raise RuntimeError(
            f"Checkpoint in {CHECKPOINT_LOCAL_DIR} is incomplete. Expected adapter_config.json and adapter_model.*"
        )

CHECKPOINT_PATH = str(CHECKPOINT_LOCAL_DIR)
print(f"Loading model: {BASE_MODEL}")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

config = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
if hasattr(config, "auto_map") and "AutoModelForCausalLM" in config.auto_map:
    model_class = get_class_from_dynamic_module(config.auto_map["AutoModelForCausalLM"], BASE_MODEL)
else:
    model_class = AutoModelForCausalLM._model_mapping[type(config)]


def _safe_init_weights(self, module):
    std = (
        self.config.initializer_range
        if hasattr(self.config, "initializer_range")
        else self.config.text_config.initializer_range
    )

    if hasattr(module, "class_embedding"):
        if module.class_embedding.data.is_floating_point():
            module.class_embedding.data.normal_(mean=0.0, std=std)

    if isinstance(module, (nn.Linear, nn.Conv2d)):
        weight = getattr(module, "weight", None)
        if weight is not None and weight.data.is_floating_point():
            weight.data.normal_(mean=0.0, std=std)
        bias = getattr(module, "bias", None)
        if bias is not None and bias.data.is_floating_point():
            bias.data.zero_()
    elif isinstance(module, nn.Embedding):
        weight = getattr(module, "weight", None)
        if weight is not None and weight.data.is_floating_point():
            weight.data.normal_(mean=0.0, std=std)
            if module.padding_idx is not None:
                weight.data[module.padding_idx].zero_()


model_class._init_weights = _safe_init_weights

if torch.cuda.is_available():
    device = "cuda"
    load_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.bfloat16,
        "device_map": {"": 0},
        "attn_implementation": "eager",
        "quantization_config": quantization_config,
    }
else:
    device = "cpu"
    load_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.float32,
        "attn_implementation": "eager",
    }

try:
    model = model_class.from_pretrained(BASE_MODEL, **load_kwargs)
except Exception as exc:
    msg = str(exc)
    if "frozenset" in msg or "bitsandbytes" in msg.lower():
        print("4-bit load failed; retrying without quantization_config")
        fallback_kwargs = {
            "trust_remote_code": True,
            "attn_implementation": "eager",
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        }
        if torch.cuda.is_available():
            fallback_kwargs["device_map"] = {"": 0}
        model = model_class.from_pretrained(BASE_MODEL, **fallback_kwargs)
    else:
        raise

if device == "cpu":
    model.to("cpu")

model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "base_img_size"):
    processor.image_processor.base_img_size = MODEL_IMAGE_SIZE
    print(f"Set processor.image_processor.base_img_size={MODEL_IMAGE_SIZE}")
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id

# -----------------------------
# 6) Read from Drive and print outputs
# -----------------------------
image_path, prompt_path = wait_for_inputs_from_public_drive(INPUT_FOLDER_URL, LOCAL_INPUT_DIR)
user_prompt = prompt_path.read_text(encoding="utf-8").strip()
image = Image.open(image_path).convert("RGB")

print(f"Using image:  {image_path}")
print(f"Using prompt: {prompt_path}")

som_annotated_image, candidate_bboxes = build_som_candidates(image)
instruction = INSTRUCTION_TEMPLATE.replace("{task_prompt}", user_prompt)
full_prompt = f"<image_start><image><image_end>\n{instruction}"

convs = [
    {"role": "system", "content": "You are agent that can see, talk and act."},
    {"role": "user", "content": full_prompt},
]
formatted_prompt = processor.tokenizer.apply_chat_template(convs, tokenize=False, add_generation_prompt=True)

if hasattr(model, "config") and getattr(model.config, "mm_use_image_start_end", False):
    formatted_prompt = formatted_prompt.replace("<image>", "<image_start><image><image_end>")

print("=" * 60)
print("PROMPT SENT TO MODEL (formatted_prompt):")
print(formatted_prompt)
print("=" * 60)

inputs = processor(images=[som_annotated_image], texts=formatted_prompt, return_tensors="pt")
inputs["pixel_values"] = inputs["pixel_values"].unsqueeze(0)
inputs["image_sizes"] = inputs["image_sizes"].unsqueeze(0)

if device == "cuda":
    inputs = inputs.to(torch.bfloat16).to("cuda")
else:
    inputs = inputs.to("cpu")

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        do_sample=False,
        num_beams=1,
        max_new_tokens=256,
        use_cache=True,
    )

generate_ids = output_ids[:, inputs["input_ids"].shape[-1] :]
response = processor.decode(generate_ids[0], skip_special_tokens=True).strip()
prediction = parse_action(response)

predicted_action = str(prediction.get("ACTION", "")).upper()
mark_id = prediction.get("MARK")
coordinate = None
if mark_id in candidate_bboxes:
    x1, y1, x2, y2 = candidate_bboxes[mark_id]
    coordinate = {"x": int((x1 + x2) / 2), "y": int((y1 + y2) / 2)}

mark_coordinate_json = {
    "mark_id": mark_id,
    "coordinate": coordinate,
    "predicted_action": predicted_action,
}

timestamp = int(time.time())
som_output_path = LOCAL_OUTPUT_DIR / f"som_annotated_{timestamp}.png"
som_annotated_image.save(som_output_path)

action_output_path = None
action_image = None
if coordinate is not None:
    action_image = som_annotated_image.copy()
    draw = ImageDraw.Draw(action_image)
    x, y = coordinate["x"], coordinate["y"]
    r = 12
    draw.ellipse((x - r, y - r, x + r, y + r), outline="lime", width=3)
    draw.line((x - 16, y, x + 16, y), fill="lime", width=3)
    draw.line((x, y - 16, x, y + 16), fill="lime", width=3)
    action_output_path = LOCAL_OUTPUT_DIR / f"action_target_{timestamp}.png"
    action_image.save(action_output_path)

print("=" * 60)
print("MODEL OUTPUT:")
print(response)
print("=" * 60)
print("MARK_ID-COORDINATE JSON:")
print(json.dumps(mark_coordinate_json, indent=2))
output_json_string = json.dumps(mark_coordinate_json)
print("OUTPUT_JSON_STRING:")
print(output_json_string)
print("=" * 60)
print(f"Saved SoM image: {som_output_path}")
if action_output_path is not None:
    print(f"Saved action-target image: {action_output_path}")
else:
    print("No action-target image saved (mark_id not found in candidate boxes).")

executor_json_path = EXECUTOR_OUTPUT_DIR / "model_output_latest.json"
executor_json_path.write_text(output_json_string + "\n", encoding="utf-8")
print(f"Wrote executor JSON: {executor_json_path}")

print("=" * 60)
print("DISPLAYING OUTPUT IMAGE:")
if action_image is not None:
    display(action_image)
else:
    display(som_annotated_image)

Installing Magma-compatible transformers fork...
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB
Applied torch.sum bool-tensor compatibility patch
Project root: /content/Magma
SOM utilities loaded from project
Applied in-cell MarkHelper font fix (system DejaVuSans)
https://drive.google.com/drive/folders/1_Wp2wBEH1-aUt9QzMmcDVCIIbLw11fR_?usp=drive_link
Checkpoint download attempt 1/3 (use_cookies=False)


Retrieving folder contents


Processing file 1thtsKUrR4XRld9a8p0NPgsOSLScYxxVF adapter_config.json
Processing file 1mMq9WD83rJgcXTpkbX8LGKRFmnU5T4oe adapter_model.safetensors
Processing file 1nxlugSugOp7O78SUiOnVgAwXLETM7F9b optimizer.pt
Processing file 1cOrl0gG3jMqAjbbTZ7M7Klr2PTT1Hlep README.md
Processing file 1Zvf1M6z9ujXQPz8pJoqhNIrzTkm6VoqI rng_state.pth
Processing file 1WY8G7dEcqrx-g_ktxcZzBQP5o1CmBll3 scheduler.pt
Processing file 1uK2uOweC8Dr0v7HHXCQ07yvBUiAx-G1r special_tokens_map.json
Processing file 1h_jRA1me6oLa3qMzfwIc75RTtvXragnj tokenizer_config.json
Processing file 1pf4_5TlovtYBBaH6jEKWIraLC12mzhjV tokenizer.json
Processing file 1kcbh-H6dqFbAVgopUdrW9OVpuS6yespc trainer_state.json
Processing file 19WvxwLtK9niTDUII7AMwrYNVZIAXmFv5 training_args.bin


Retrieving folder contents completed
Building directory structure
Building directory structure completed


Checkpoint download failed on attempt 1: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1thtsKUrR4XRld9a8p0NPgsOSLScYxxVF

but Gdown can't. Please check connections and permissions.
Checkpoint download attempt 2/3 (use_cookies=False)


...

Checkpoint download failed on attempt 3: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1thtsKUrR4XRld9a8p0NPgsOSLScYxxVF

but Gdown can't. Please check connections and permissions.
Retrying checkpoint download with use_cookies=True ...


Retrieving folder contents


Processing file 1thtsKUrR4XRld9a8p0NPgsOSLScYxxVF adapter_config.json
Processing file 1mMq9WD83rJgcXTpkbX8LGKRFmnU5T4oe adapter_model.safetensors
Processing file 1nxlugSugOp7O78SUiOnVgAwXLETM7F9b optimizer.pt
Processing file 1cOrl0gG3jMqAjbbTZ7M7Klr2PTT1Hlep README.md
Processing file 1Zvf1M6z9ujXQPz8pJoqhNIrzTkm6VoqI rng_state.pth
Processing file 1WY8G7dEcqrx-g_ktxcZzBQP5o1CmBll3 scheduler.pt
Processing file 1uK2uOweC8Dr0v7HHXCQ07yvBUiAx-G1r special_tokens_map.json
Processing file 1h_jRA1me6oLa3qMzfwIc75RTtvXragnj tokenizer_config.json
Processing file 1pf4_5TlovtYBBaH6jEKWIraLC12mzhjV tokenizer.json
Processing file 1kcbh-H6dqFbAVgopUdrW9OVpuS6yespc trainer_state.json
Processing file 19WvxwLtK9niTDUII7AMwrYNVZIAXmFv5 training_args.bin


Retrieving folder contents completed
Building directory structure
Building directory structure completed


KeyboardInterrupt: 

In [ ]:
# Debug print: actual image/tensor sizes used for model input
if 'som_annotated_image' in globals():
    print('Original PIL image size (W, H):', som_annotated_image.size)
else:
    print('som_annotated_image not found. Run Cell 1 first.')

if 'inputs' in globals():
    print('inputs["pixel_values"].shape:', tuple(inputs['pixel_values'].shape))
    if 'image_sizes' in inputs:
        print('inputs["image_sizes"]:', inputs['image_sizes'])
        try:
            print('inputs["image_sizes"].shape:', tuple(inputs['image_sizes'].shape))
        except Exception:
            pass
else:
    print('inputs not found. Run Cell 1 first.')